# Vortex-50M - Cloud Training Notebook

From-scratch **<=50M parameter** decoder-only Transformer, **English-only** vocab.

```
49,497,216 params  |  640d x 12L  |  10Q/2KV GQA  |  vocab 8,192 (tied)
11% embedding  /  89% transformer layers
```

This notebook runs the whole pipeline on a fresh cloud GPU:
**tokenizer -> retokenize -> verify -> train -> evaluate -> push**.

> **Order matters.** The old `.bin` shards were tokenized with Qwen3's 151,670-token
> vocab and are **invalid** for this model. Retrain the tokenizer and rebuild the
> shards before training, or the run wastes days on garbage.

Before you start: set `HF_TOKEN` and update the repo IDs in the config cell.

## 1. Runtime setup

In [ ]:
import os, sys, json, subprocess, textwrap, pathlib, shutil, time
from pathlib import Path

def run(cmd, check=True, cwd=None, env=None):
    print(f"\n$ {cmd}", flush=True)
    return subprocess.run(cmd, shell=True, check=check, cwd=cwd, env=env)

print("Python:", sys.version.split()[0])
run("nvidia-smi", check=False)

# A100 40GB / L4 24GB / T4 16GB all work; see the batch notes in section 5b.
print("\nNotebook ready.")

## 2. Get the repo

If the repo is already in the runtime, set `CLONE_REPO = False`.

In [ ]:
CLONE_REPO = True
GIT_REPO_URL = "https://github.com/OEvortex/VTX-300M.git"
REPO_DIR = Path("/content/VTX-300M")

if CLONE_REPO:
    if not REPO_DIR.exists():
        run(f"git clone {GIT_REPO_URL} {REPO_DIR}")
    else:
        print(f"Repo already present: {REPO_DIR}")
else:
    REPO_DIR = Path.cwd()

SRC_DIR = REPO_DIR / "src"
assert (SRC_DIR / "pretrain.py").exists(), f"src/pretrain.py not found under {REPO_DIR}"
print("Repo:  ", REPO_DIR)
print("Source:", SRC_DIR)

## 3. Install dependencies

In [ ]:
INSTALL_TORCH = True       # set False if the image already has a working CUDA torch
INSTALL_TOKENIZERS = True   # train_tokenizer.py needs this
INSTALL_EVAL = True        # benchmark deps (HellaSwag / ARC / PIQA / WinoGrande)

run(f"{sys.executable} -m pip install -U pip setuptools wheel")

if INSTALL_TORCH:
    run(f"{sys.executable} -m pip install -U torch --index-url https://download.pytorch.org/whl/cu124")

base = [
    "transformers>=4.45.0",
    "huggingface_hub>=0.24.0",
    "hf_transfer",
    "safetensors",
    "numpy",
    "scipy",
    "trackio",
]
if INSTALL_TOKENIZERS:
    base.append("tokenizers")
if INSTALL_EVAL:
    base += ["datasets", "accelerate"]
run(f"{sys.executable} -m pip install -U " + " ".join(base))

import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)
print("cuda available:", torch.cuda.is_available())

## 4. Hugging Face login

Needs **write** access to create/push the model, tokenizer and data repos.

In [ ]:
from getpass import getpass
from huggingface_hub import login, whoami, create_repo

HF_TOKEN = os.environ.get("HF_TOKEN") or getpass("HF_TOKEN: ")
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

login(token=HF_TOKEN, add_to_git_credential=False)
print(whoami(token=HF_TOKEN))

## 5. Configuration

Smoke test: `STEPS = 5`, `PUSH_EVERY = 5`. Real run: raise `STEPS` per section 5b.

In [ ]:
# -- Repos ---------------------------------------------------------------
MODEL_REPO_ID    = "VTXAI/vortex-50m"
TOKENIZER_REPO  = "VTXAI/vortex-tok-8k"
DATA_REPO_ID    = "VTXAI/vortex-50m-data-8k"
TRACKIO_SPACE_ID = "VTXAI/vortex-50m-trackio"   # None disables Trackio

# -- Architecture preset (see config.PRESETS) ----------------------------
ARCH = "vortex-50m"          # 49.50M | 640d x 12L | 10Q/2KV | vocab 8,192
TOKENIZER_VOCAB = 8192        # must match the trained tokenizer exactly
CONTEXT = 2048

# -- Data ----------------------------------------------------------------
RAW_CORPUS    = "/content/corpus"          # your .jsonl / .txt / .parquet English data
TOKENIZER_DIR = "/content/vortex-tok-8k"
DATA_DIR      = "/content/data8k"
SAVE_DIR      = "/content/vortex_50m_ckpt"

# -- Training ------------------------------------------------------------
STEPS, WARMUP = 60000, 600
LR, MIN_LR    = 6e-4, 6e-5
BATCH, GRAD_ACCUM = 8, 4    # eff. batch 32 | 65,536 tok/step | ~3.9B tokens
PUSH_EVERY, LOG_EVERY = 3000, 25
COMPILE = True
TRACKIO_PROJECT = "vortex-50m"

train_config = {
    "arch": ARCH, "tokenizer": TOKENIZER_DIR, "shards": "AUTO",
    "steps": STEPS, "warmup": WARMUP, "lr": LR, "min_lr": MIN_LR,
    "weight_decay": 0.1, "beta1": 0.9, "beta2": 0.99, "grad_clip": 1.0,
    "batch": BATCH, "grad_accum": GRAD_ACCUM, "block": CONTEXT, "seed": 42,
    "hub_repo": MODEL_REPO_ID, "push_every": PUSH_EVERY, "log_every": LOG_EVERY,
    "trackio_space": TRACKIO_SPACE_ID, "trackio_project": TRACKIO_PROJECT,
    "save_dir": SAVE_DIR, "compile": COMPILE,
}
CONFIG_PATH = SRC_DIR / "train_config.json"
CONFIG_PATH.write_text(json.dumps(train_config, indent=2))
print(json.dumps(train_config, indent=2))
print("\n->", CONFIG_PATH)

### 5b. Batch sizing by GPU

| GPU | VRAM | batch | grad_accum | eff. batch | tok/step |
|---|---|---|---|---|---|
| A100 / H100 | 40-80GB | 16 | 2 | 32 | 65,536 |
| RTX 5090 | 32GB | 8 | 4 | 32 | 65,536 |
| L4 / A10G | 24GB | 4 | 8 | 32 | 65,536 |
| T4 | 16GB | 2 | 16 | 32 | 65,536 |

Keep the **effective batch at 32** and trade `batch` against `grad_accum`.
If you OOM, halve `batch` and double `grad_accum`.

## 6. Train the tokenizer (English, 8,192 tokens)

This is the step that unlocks the parameter budget. The Qwen3 151,670-token vocab
costs `77.7M` parameters as an embedding table at 512d - **155% of the entire 50M
budget** before a single transformer layer exists.

An 8K English BPE costs **5.2M**, which is what lets the same budget buy a 640d-wide
model instead of more lookup table.

In [ ]:
TRAIN_TOKENIZER = True
FORCE_RETRAIN   = False   # set True to retrain even if TOKENIZER_DIR exists

already = Path(TOKENIZER_DIR) / "tokenizer.json"

if TRAIN_TOKENIZER and (FORCE_RETRAIN or not already.exists()):
    run(
        f"{sys.executable} train_tokenizer.py "
        f"{RAW_CORPUS} --out {TOKENIZER_DIR} --vocab-size {TOKENIZER_VOCAB}",
        cwd=SRC_DIR,
    )
else:
    print(f"Reusing existing tokenizer at {TOKENIZER_DIR}")

from tokenizers import Tokenizer
tok = Tokenizer.from_file(str(Path(TOKENIZER_DIR) / "tokenizer.json"))
actual_vocab = tok.get_vocab_size()
print(f"\ntokenizer vocab = {actual_vocab:,} (requested {TOKENIZER_VOCAB:,})")

if actual_vocab != TOKENIZER_VOCAB:
    print("\n[NOTE] Trained vocab differs from the preset.")
    print("       pretrain.py adopts the tokenizer's real vocab, which is")
    print("       correct -- just be aware the param count shifts slightly.")

### 6b. Measure compression before you commit

A smaller vocab compresses English less efficiently, so the same corpus yields
**more tokens**. The parameter win is certain; this cost is corpus-specific.

Roughly: 32K byte-level BPE is about 4.0-4.5 chars/token on English prose, 8K-16K
is about 3.2-3.8. Below ~3.0 means the corpus has unusual vocabulary - consider 16K.

In [ ]:
run(
    f"{sys.executable} train_tokenizer.py --out {TOKENIZER_DIR} "
    f"--stats --stats-files {RAW_CORPUS} --stats-docs 3000",
    cwd=SRC_DIR,
)

## 7. Retokenize the corpus

**Mandatory.** The existing shards encode token ids under the old 151K vocab; under
the new 8K vocab those integers are meaningless. Produces `uint32` EOS-separated
shards that `dataset.MMapDataset` reads directly.

In [ ]:
RETOKENIZE = True

if RETOKENIZE:
    run(
        f"{sys.executable} retokenize.py {RAW_CORPUS} "
        f"--tokenizer {TOKENIZER_DIR} --out {DATA_DIR} "
        f"--tokens-per-shard 100000000",
        cwd=SRC_DIR,
    )

shards = sorted(str(p) for p in Path(DATA_DIR).glob("*.bin"))
if not shards:
    raise SystemExit("No .bin shards produced - check RAW_CORPUS path.")

total = sum(Path(s).stat().st_size for s in shards) // 4   # uint32
print(f"\n{len(shards)} shard(s), {total/1e9:.2f}B tokens")
for s in shards[:5]:
    print(f"  {Path(s).name}  {Path(s).stat().st_size/1e6:8.1f} MB")

### 7b. Push tokenizer + data to the Hub (optional)

Makes the run resumable on a fresh machine without redoing the tokenizer.

In [ ]:
PUSH_ARTIFACTS = False

if PUSH_ARTIFACTS:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)

    api.create_repo(TOKENIZER_REPO, token=HF_TOKEN, exist_ok=True)
    api.upload_folder(folder_path=TOKENIZER_DIR, repo_id=TOKENIZER_REPO,
                      commit_message="english 8k tokenizer", token=HF_TOKEN)
    print("pushed tokenizer ->", TOKENIZER_REPO)

    api.create_repo(DATA_REPO_ID, repo_type="dataset", token=HF_TOKEN, exist_ok=True)
    api.upload_folder(folder_path=DATA_DIR, repo_id=DATA_REPO_ID,
                      repo_type="dataset", commit_message="retokenized shards",
                      token=HF_TOKEN)
    print("pushed data ->", DATA_REPO_ID)
else:
    print("skipping artifact push (PUSH_ARTIFACTS = False)")

## 8. Verify the architecture

Cheap, and it catches the failure modes that silently ruin a long run: over-budget
params, blocks that are not identity at init, future-token leakage, GQA head counts
that do not divide, and a broken save/load round trip.

In [ ]:
r = run(f"{sys.executable} verify_arch.py", cwd=SRC_DIR, check=False)
print(f"\nverify_arch.py exit code: {r.returncode}")
if r.returncode != 0:
    raise SystemExit("Architecture verification FAILED - do not start a long run.")
print("Architecture verified. Safe to train.")

## 9. Create the model repo

In [ ]:
create_repo(MODEL_REPO_ID, token=HF_TOKEN, private=False, exist_ok=True)
if TRACKIO_SPACE_ID:
    create_repo(TRACKIO_SPACE_ID, token=HF_TOKEN, repo_type="space",
                space_sdk="static", private=False, exist_ok=True)
print("Hub repos ready")

## 10. Train

Resumes automatically from the newest `step_*` in `SAVE_DIR`. Use `--no-resume` to
start fresh. Expect roughly `ln(8192) = 9.0` at step 0, falling toward ~3.5.

In [ ]:
env = os.environ.copy()
if TRACKIO_SPACE_ID:
    env["TRACKIO_SPACE_ID"] = TRACKIO_SPACE_ID
    env["TRACKIO_PROJECT"] = TRACKIO_PROJECT

cmd = (
    f"{sys.executable} pretrain.py --config {CONFIG_PATH.name} "
    f"--arch {ARCH} --tokenizer {TOKENIZER_DIR} "
    f"--shards {' '.join(shards)} --block {CONTEXT}"
)
print("Command:\n ", cmd, "\n")
t0 = time.time()
run(cmd, cwd=SRC_DIR, env=env)
print(f"\nTraining finished in {(time.time()-t0)/3600:.2f} h")

## 11. Find the latest checkpoint

In [ ]:
def latest_checkpoint(save_dir):
    root = Path(save_dir)
    steps = []
    for p in root.glob("step_*"):
        if p.is_dir():
            try:
                steps.append((int(p.name.split("_", 1)[1]), p))
            except Exception:
                pass
    return sorted(steps)[-1][1] if steps else None

LATEST = latest_checkpoint(SAVE_DIR)
print("Latest checkpoint:", LATEST)

## 12. Evaluate

Sanity check that the model learned English. At 50M params expect roughly
chance-to-modest on these - the point is to confirm it is far above random, not to
set a SOTA number.

In [ ]:
if INSTALL_EVAL and LATEST is not None:
    run(
        f"{sys.executable} eval_benchmarks.py --ckpt {LATEST} "
        f"--tokenizer {TOKENIZER_DIR} --limit 500",
        cwd=SRC_DIR, check=False,
    )
else:
    print("skipping eval (INSTALL_EVAL / LATEST)")

## 13. Push the final model

Training pushes every `PUSH_EVERY` steps. This cell pushes the final state.

In [ ]:
if LATEST is not None:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.upload_folder(folder_path=str(LATEST), repo_id=MODEL_REPO_ID,
                      commit_message="final", token=HF_TOKEN)
    api.upload_folder(folder_path=TOKENIZER_DIR, repo_id=MODEL_REPO_ID,
                      commit_message="tokenizer", token=HF_TOKEN)
    print("pushed ->", MODEL_REPO_ID)
else:
    print("no checkpoint found")